<a href="https://colab.research.google.com/github/Quang365/mri-recognition/blob/main/notebooks/VAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [2]:
# Dataset paths
RANGPUR_DATA_ROOT = "/home/groups/comp3710/OASIS"

TRAIN_DIR = os.path.join(RANGPUR_DATA_ROOT, "keras_png_slices_train")
VAL_DIR = os.path.join(RANGPUR_DATA_ROOT, "keras_png_slices_validate")
TEST_DIR = os.path.join(RANGPUR_DATA_ROOT, "keras_png_slices_test")

print("Dataset root:", RANGPUR_DATA_ROOT)

if os.path.exists(RANGPUR_DATA_ROOT):
    print("OASIS dataset found.")
    print("Training directory:", TRAIN_DIR)
    print("Validation directory:", VAL_DIR)
    print("Test directory:", TEST_DIR)
else:
    print("OASIS dataset not available in this environment.")
    print("The dataset will be loaded when this notebook is run on Rangpur.")

Dataset root: /home/groups/comp3710/OASIS
OASIS dataset not available in this environment.
The dataset will be loaded when this notebook is run on Rangpur.


In [3]:
class MRIDataset(Dataset):
    """
    Dataset for loading preprocessed OASIS MRI slices.
    Each image is loaded as a single-channel grayscale tensor.
    """

    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform

        if os.path.exists(image_dir):
            self.image_files = sorted([
                f for f in os.listdir(image_dir)
                if f.lower().endswith(".png")
            ])
        else:
            self.image_files = []

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        image_path = os.path.join(
            self.image_dir,
            self.image_files[idx]
        )

        image = Image.open(image_path).convert("L")

        if self.transform:
            image = self.transform(image)

        return image

In [4]:
image_transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = MRIDataset(TRAIN_DIR, transform=image_transform)
val_dataset = MRIDataset(VAL_DIR, transform=image_transform)
test_dataset = MRIDataset(TEST_DIR, transform=image_transform)

print(f"Training images:   {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print(f"Testing images:    {len(test_dataset)}")

Training images:   0
Validation images: 0
Testing images:    0
